# NSW Active-Fire Reliability Pilot

> **Important Interpretation Callout:** This notebook performs a spatiotemporal reliability audit matching satellite hotspot observations (DEA Hotspots) against official post-event fire boundary records (NPWS Fire History). This is a *calibration and reliability study* of spatial-temporal overlap, **not a detector-accuracy evaluation**. Unmatched observations are labelled as *unresolved*, and must not be assumed to be errors or sensor inaccuracies without independent ground truth.

### Setup and Environment Check

We verify standard package versions to ensure reproducible executions.

In [ ]:
# Execution Configuration
EXECUTION_MODE = "snapshot"  # Options: "snapshot", "live_refresh"
SNAPSHOT_SLUG = "tuannm3812/nsw-active-fire-pilot-snapshot"

# Spatiotemporal Matching Configurations
TEMPORAL_GRACE_DAYS = 1.0  # Temporal window tolerance in days
DISPLAY_SAMPLE_SIZE = 1000  # Number of hotspots to display on pilot map
RANDOM_SEED = 36126        # Random seed for reproducibility

# Standard Library Imports
import json
import hashlib
import os
import sys
from pathlib import Path
from datetime import datetime, timedelta, timezone
import math
from typing import Dict, List, Tuple, Optional, Sequence, Iterable

# Third-Party Library Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

from importlib.metadata import version
print("Runtime Environment:")
for pkg in ["pandas", "numpy", "matplotlib", "colorspacious", "nbformat"]:
    try:
        print(f"- {pkg}: {version(pkg)}")
    except Exception:
        print(f"- {pkg}: not installed")


## 1. Project overview

This project evaluates the spatiotemporal reliability of satellite active-fire hotspots in New South Wales (NSW), Australia. Satellite-derived hotspot products (such as those from MODIS, VIIRS, and AHI) are widely used for real-time fire detection, but their operational reliability must be calibrated against official historical fire boundaries to understand spatial and temporal alignment. We focus on a bounded region of NSW during a fortnight of intense fire activity in January 2020.

## 2. Methodology & Mathematical Formulation

To calibrate the spatiotemporal overlap between satellite hotspot observations and historical fire boundaries, we define a formal matching framework. Let a hotspot observation $h$ be represented as:
$$h = (\phi_h, \lambda_h, t_h, \text{sensor}(h))$$
where $(\phi_h, \lambda_h)$ is the geographic position (latitude, longitude), $t_h$ is the acquisition timestamp, and $\text{sensor}(h)$ designates the observing instrument (e.g., MODIS, VIIRS, AHI).

A fire event $F_i$ from the historical boundary record is represented as:
$$F_i = (P_i, [t_{\text{ignition}, i}, t_{\text{extinguish}, i}])$$
where $P_i$ is the spatial polygon (or MultiPolygon) boundary, and $[t_{\text{ignition}, i}, t_{\text{extinguish}, i}]$ is the active burn interval.

We evaluate matching under two distinct regimes:

### A. Exact Spatial Matching (Baseline)
A hotspot $h$ is classified as an **exact match** to fire event $F_i$ if it satisfies both spatial containment and temporal window overlap (including a symmetric temporal grace period $\Delta t$):
1. **Spatial Containment:**
   $$(\phi_h, \lambda_h) \in P_i$$
2. **Temporal Alignment:**
   $$t_h \in [t_{\text{ignition}, i} - \Delta t, t_{\text{extinguish}, i} + \Delta t]$$
We set the temporal grace period to $\Delta t = 1 \text{ day}$ to account for reporting latencies and ignition/extinguishment boundary uncertainties.

### B. Sensor-Buffered Spatial Matching
To account for the physical limits and positional accuracy of satellite sensors, we expand the spatial boundary $P_i$ using a sensor-specific buffer $\epsilon_{s}$. A hotspot $h$ matches $F_i$ under buffered tolerances if:
1. **Buffered Spatial Containment:**
   $$\text{dist}((\phi_h, \lambda_h), P_i) \le \epsilon_{s}$$
   where $\text{dist}$ is the shortest spherical distance from the hotspot coordinates to the boundary of $P_i$:
   $$\text{dist}(h_{\text{pos}}, P_i) = \begin{cases} 0 & \text{if } h_{\text{pos}} \in P_i \\ \min_{p \in \partial P_i} \text{dist}_{\text{great-circle}}(h_{\text{pos}}, p) & \text{otherwise} \end{cases}$$
2. **Temporal Alignment:**
   $$t_h \in [t_{\text{ignition}, i} - \Delta t, t_{\text{extinguish}, i} + \Delta t]$$

The spatial buffer threshold $\epsilon_{s}$ is determined by the sensor's native nominal spatial resolution:
* **VIIRS:** $\epsilon_{\text{VIIRS}} = 0.375\text{ km}$ (high-resolution channels)
* **MODIS:** $\epsilon_{\text{MODIS}} = 1.0\text{ km}$
* **AHI:** $\epsilon_{\text{AHI}} = 2.0\text{ km}$

### 2.1 Spatiotemporal Containment Core Algorithms

The core mathematical and containment logic for ray-casting containment checks and segment-distance buffer calculations.

In [ ]:
from datetime import datetime, timedelta, timezone
import math
from typing import Dict, Iterable, Iterator, List, Optional, Sequence, Tuple, Union


Point = Tuple[float, float]


def _coordinate_points(coordinates: Sequence) -> Iterator[Point]:
    if not coordinates:
        return
    if isinstance(coordinates[0], (int, float)):
        yield float(coordinates[0]), float(coordinates[1])
        return
    for part in coordinates:
        yield from _coordinate_points(part)


def geometry_bounds(geometry: Dict) -> Tuple[float, float, float, float]:
    points = list(_coordinate_points(geometry.get("coordinates", [])))
    if not points:
        raise ValueError("Geometry has no coordinates")
    longitudes = [point[0] for point in points]
    latitudes = [point[1] for point in points]
    return min(longitudes), min(latitudes), max(longitudes), max(latitudes)


def prepare_features(features: Iterable[Dict]) -> List[Dict]:
    prepared = []
    for feature in features:
        copy = dict(feature)
        copy["_bbox"] = geometry_bounds(copy["geometry"])
        prepared.append(copy)
    return prepared


def _point_in_bbox(lon: float, lat: float, bbox: Tuple[float, float, float, float]) -> bool:
    return bbox[0] <= lon <= bbox[2] and bbox[1] <= lat <= bbox[3]


def _point_segment_distance_km(point: Point, start: Point, end: Point) -> float:
    lon_scale = 111.320 * math.cos(math.radians(point[1]))
    lat_scale = 110.574
    ax = (start[0] - point[0]) * lon_scale
    ay = (start[1] - point[1]) * lat_scale
    bx = (end[0] - point[0]) * lon_scale
    by = (end[1] - point[1]) * lat_scale
    segment_x = bx - ax
    segment_y = by - ay
    denominator = segment_x * segment_x + segment_y * segment_y
    if denominator == 0:
        return math.hypot(ax, ay)
    projection = max(0.0, min(1.0, -(ax * segment_x + ay * segment_y) / denominator))
    return math.hypot(ax + projection * segment_x, ay + projection * segment_y)


def point_within_geometry_buffer(
    lon: float, lat: float, geometry: Dict, buffer_km: float
) -> bool:
    if point_in_geometry(lon, lat, geometry):
        return True
    point = (float(lon), float(lat))
    minimum = float("inf")
    coordinates = geometry.get("coordinates", [])
    polygons = [coordinates] if geometry.get("type") == "Polygon" else coordinates
    for polygon in polygons:
        for ring in polygon:
            for index in range(len(ring)):
                start = (float(ring[index - 1][0]), float(ring[index - 1][1]))
                end = (float(ring[index][0]), float(ring[index][1]))
                minimum = min(minimum, _point_segment_distance_km(point, start, end))
                if minimum <= buffer_km:
                    return True
    return False


def _point_on_segment(point: Point, start: Point, end: Point, tolerance: float = 1e-10) -> bool:
    px, py = point
    x1, y1 = start
    x2, y2 = end
    cross = (px - x1) * (y2 - y1) - (py - y1) * (x2 - x1)
    if abs(cross) > tolerance:
        return False
    return (
        min(x1, x2) - tolerance <= px <= max(x1, x2) + tolerance
        and min(y1, y2) - tolerance <= py <= max(y1, y2) + tolerance
    )


def _point_in_ring(point: Point, ring: Sequence[Sequence[float]]) -> bool:
    inside = False
    for index in range(len(ring)):
        start = (float(ring[index - 1][0]), float(ring[index - 1][1]))
        end = (float(ring[index][0]), float(ring[index][1]))
        if _point_on_segment(point, start, end):
            return True
        x1, y1 = start
        x2, y2 = end
        crosses = (y1 > point[1]) != (y2 > point[1])
        if crosses:
            intersection_x = (x2 - x1) * (point[1] - y1) / (y2 - y1) + x1
            if point[0] < intersection_x:
                inside = not inside
    return inside


def _point_in_polygon(point: Point, polygon: Sequence[Sequence[Sequence[float]]]) -> bool:
    if not polygon or not _point_in_ring(point, polygon[0]):
        return False
    return not any(_point_in_ring(point, hole) for hole in polygon[1:])


def point_in_geometry(lon: float, lat: float, geometry: Dict) -> bool:
    """Return whether a WGS84 point lies in a GeoJSON Polygon or MultiPolygon."""
    point = (float(lon), float(lat))
    geometry_type = geometry.get("type")
    coordinates = geometry.get("coordinates", [])
    if geometry_type == "Polygon":
        return _point_in_polygon(point, coordinates)
    if geometry_type == "MultiPolygon":
        return any(_point_in_polygon(point, polygon) for polygon in coordinates)
    raise ValueError("Only Polygon and MultiPolygon geometries are supported")


def parse_datetime(value: Optional[Union[str, int, float]]) -> Optional[datetime]:
    if value in (None, ""):
        return None
    if isinstance(value, (int, float)):
        return datetime.fromtimestamp(value / 1000, tz=timezone.utc)
    text = str(value).strip().replace("Z", "+00:00")
    parsed = datetime.fromisoformat(text)
    if parsed.tzinfo is None:
        parsed = parsed.replace(tzinfo=timezone.utc)
    return parsed.astimezone(timezone.utc)


def within_event_window(observed_at: datetime, properties: Dict, grace_days: int) -> bool:
    """Check an observation against ignition/extinguish dates plus symmetric grace."""
    ignition = parse_datetime(properties.get("ignition_date"))
    if ignition is None:
        return False
    extinguish = parse_datetime(properties.get("extinguish_date")) or ignition
    grace = timedelta(days=grace_days)
    observed = parse_datetime(observed_at)
    return ignition - grace <= observed <= extinguish + grace


def _normalized_fire_type(value: Optional[str]) -> str:
    normalized = (value or "").strip().lower().replace(" ", "_")
    if normalized in {"bushfire", "wildfire", "wild_fire"}:
        return "bushfire"
    if normalized in {"prescribed_burn", "hazard_reduction", "planned_burn"}:
        return "prescribed_burn"
    return "other_fire"


def classify_hotspot(
    hotspot: Dict,
    features: Iterable[Dict],
    grace_days: int = 1,
    spatial_buffer_km: float = 0.0,
) -> Dict:
    """Classify one hotspot using polygon containment and the event date window."""
    observed_at = parse_datetime(hotspot["datetime"])
    spatial_matches: List[Dict] = []
    temporal_matches: List[Dict] = []
    for feature in features:
        if feature.get("_bbox"):
            degree_buffer = spatial_buffer_km / 80.0
            bbox = feature["_bbox"]
            expanded = (
                bbox[0] - degree_buffer,
                bbox[1] - degree_buffer,
                bbox[2] + degree_buffer,
                bbox[3] + degree_buffer,
            )
            if not _point_in_bbox(
                float(hotspot["longitude"]), float(hotspot["latitude"]), expanded
            ):
                continue
        if point_within_geometry_buffer(
            hotspot["longitude"],
            hotspot["latitude"],
            feature["geometry"],
            spatial_buffer_km,
        ):
            spatial_matches.append(feature)
            if within_event_window(observed_at, feature.get("properties", {}), grace_days):
                temporal_matches.append(feature)

    if temporal_matches:
        chosen = sorted(
            temporal_matches,
            key=lambda feature: (
                parse_datetime(feature.get("properties", {}).get("ignition_date"))
                or datetime.max.replace(tzinfo=timezone.utc),
                str(feature.get("properties", {}).get("fire_id") or ""),
            ),
        )[0]
        properties = chosen.get("properties", {})
        return {
            **hotspot,
            "match_class": _normalized_fire_type(properties.get("fire_type")),
            "fire_id": properties.get("fire_id"),
            "fire_name": properties.get("fire_name"),
            "spatial_match_count": len(spatial_matches),
            "temporal_match_count": len(temporal_matches),
        }

    return {
        **hotspot,
        "match_class": "spatial_only" if spatial_matches else "unmatched",
        "fire_id": None,
        "fire_name": None,
        "spatial_match_count": len(spatial_matches),
        "temporal_match_count": 0,
    }

### 2.2 Data Aggregation and Metrics Helpers

Helper functions to compute statistics, match rates, event concentration distributions, and refresh differences.

In [ ]:
import pandas as pd
import numpy as np


def headline_summary(exact: pd.DataFrame, buffered: pd.DataFrame) -> dict:
    total_exact = len(exact)
    total_buffered = len(buffered)
    if total_exact != total_buffered:
        raise ValueError("Exact and buffered dataframes must have the same length")
        
    exact_matches = sum(exact["match_class"] != "unmatched")
    exact_unresolved = total_exact - exact_matches
    
    buffered_matches = sum(buffered["match_class"].isin(["bushfire", "prescribed_burn", "other_fire"]))
    buffered_unresolved = total_buffered - buffered_matches
    
    return {
        "total_hotspots": total_exact,
        "exact_matches": int(exact_matches),
        "exact_unresolved": int(exact_unresolved),
        "exact_match_rate": float(exact_matches / total_exact) if total_exact > 0 else 0.0,
        "buffered_matches": int(buffered_matches),
        "buffered_unresolved": int(buffered_unresolved),
        "buffered_match_rate": float(buffered_matches / total_buffered) if total_buffered > 0 else 0.0,
    }


def sensor_summary(matches: pd.DataFrame) -> pd.DataFrame:
    summary_list = []
    grouped = matches.groupby("sensor")
    for sensor, group in grouped:
        total = len(group)
        matched = sum(group["match_class"].isin(["bushfire", "prescribed_burn", "other_fire"]))
        unresolved = total - matched
        summary_list.append({
            "sensor": sensor,
            "total_hotspots": total,
            "matched_hotspots": int(matched),
            "unresolved_hotspots": int(unresolved),
            "match_rate": float(matched / total) if total > 0 else 0.0
        })
    return pd.DataFrame(summary_list)


def event_concentration(matches: pd.DataFrame) -> pd.DataFrame:
    # Filter for matched hotspots only
    matched = matches[matches["match_class"].isin(["bushfire", "prescribed_burn", "other_fire"])]
    if matched.empty:
        return pd.DataFrame(columns=["fire_name", "fire_id", "matched_hotspots", "percentage"])
        
    total_matched = len(matched)
    # Group by fire_name and fire_id (or OBJECTID)
    grouped = matched.groupby(["fire_name", "fire_id"], dropna=False)
    summary_list = []
    for (fire_name, fire_id), group in grouped:
        count = len(group)
        summary_list.append({
            "fire_name": fire_name if pd.notna(fire_name) else "Unknown",
            "fire_id": fire_id if pd.notna(fire_id) else "Unknown",
            "matched_hotspots": count,
            "percentage": float(count / total_matched) if total_matched > 0 else 0.0
        })
    df = pd.DataFrame(summary_list)
    return df.sort_values(by="matched_hotspots", ascending=False).reset_index(drop=True)


def deterministic_display_sample(frame: pd.DataFrame, size: int, seed: int = 42) -> pd.DataFrame:
    n = min(len(frame), size)
    if n <= 0:
        return frame.copy()
    return frame.sample(n=n, random_state=seed).sort_index()


def compare_refresh(reviewed: dict, refreshed: dict) -> pd.DataFrame:
    comparison_list = []
    for key in reviewed:
        rev_val = reviewed[key]
        ref_val = refreshed.get(key)
        status = "stable" if rev_val == ref_val else "changed"
        comparison_list.append({
            "metric": key,
            "reviewed": rev_val,
            "refreshed": ref_val,
            "status": status
        })
    return pd.DataFrame(comparison_list)


def assert_snapshot_invariants(actual: dict, expected: dict) -> None:
    differences = {key: (expected[key], actual.get(key)) for key in expected if actual.get(key) != expected[key]}
    if differences:
        raise AssertionError(f"Snapshot invariants changed: {differences}")


### 2.3 Visualization Helpers

Functions utilizing the Okabe-Ito colorblind-accessible color palette and Matplotlib version-compatibility checks to render analysis figures.

In [ ]:
import sys
import matplotlib
if "ipykernel" not in sys.modules:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from typing import Dict, List

# Colorblind-friendly palette (Okabe-Ito)
OKABE_ITO_COLORS = ["#0072B2", "#E69F00", "#CC79A7", "#999999"]
SENSOR_COLORS = {
    "VIIRS": "#0072B2",
    "MODIS": "#E69F00",
    "AHI": "#CC79A7",
    "unmatched": "#999999",
    "unresolved": "#999999"
}

def _apply_premium_style(ax: plt.Axes, title: str, xlabel: str = "", ylabel: str = "") -> None:
    """Applies clean, publication-quality theme settings to Matplotlib axes."""
    ax.set_facecolor("white")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_color("#333333")
    ax.spines["bottom"].set_color("#333333")
    ax.spines["left"].set_linewidth(1.0)
    ax.spines["bottom"].set_linewidth(1.0)
    
    ax.tick_params(colors="#333333", labelsize=9, width=1.0)
    if title:
        ax.set_title(title, color="black", fontsize=11, fontweight="bold", pad=16, loc="left")
    if xlabel:
        ax.set_xlabel(xlabel, color="#333333", fontsize=9.5, labelpad=8)
    if ylabel:
        ax.set_ylabel(ylabel, color="#333333", fontsize=9.5, labelpad=8)


def plot_sensor_composition(frame: pd.DataFrame) -> plt.Figure:
    """Plot the total count of hotspots by sensor."""
    fig, ax = plt.subplots(figsize=(6.5, 4.0), facecolor="white")
    
    df = frame.sort_values(by="total_hotspots", ascending=True)
    total_n = df["total_hotspots"].sum()
    
    # Map colors to sensors
    colors = [SENSOR_COLORS.get(sensor, OKABE_ITO_COLORS[0]) for sensor in df["sensor"]]
    
    bars = ax.barh(df["sensor"], df["total_hotspots"], color=colors, height=0.55)
    _apply_premium_style(ax, f"Active-Fire Hotspots by Sensor (Total N={total_n:,})", "Total Hotspot Observations")
    
    # Grid lines only vertical, very light
    ax.grid(True, axis="x", linestyle=":", alpha=0.4, color="#CCCCCC", zorder=0)
    ax.set_axisbelow(True)
    
    # Value labels
    for bar in bars:
        width = bar.get_width()
        pct = (width / total_n) * 100.0
        ax.text(width + (total_n * 0.01), bar.get_y() + bar.get_height()/2, 
                f"{int(width):,} ({pct:.1f}%)", 
                va="center", ha="left", color="black", fontsize=8.5, fontweight="semibold")
                
    # Extra right padding
    ax.set_xlim(0, max(df["total_hotspots"]) * 1.15)
    fig.tight_layout()
    return fig


def plot_match_rates(frame: pd.DataFrame) -> plt.Figure:
    """Plot match rate by sensor with denominators in the labels."""
    fig, ax = plt.subplots(figsize=(7.0, 4.0), facecolor="white")
    
    labels = []
    rates = []
    colors = []
    for _, row in frame.iterrows():
        sensor = row["sensor"]
        n = int(row["total_hotspots"])
        rate = float(row["match_rate"])
        labels.append(f"{sensor}\n(n={n:,})")
        rates.append(rate * 100.0)
        colors.append(SENSOR_COLORS.get(sensor, OKABE_ITO_COLORS[1]))
        
    y_pos = np.arange(len(labels))
    bars = ax.barh(y_pos, rates, color=colors, height=0.55)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(labels, color="black", fontsize=9)
    
    _apply_premium_style(ax, "Spatiotemporal Match Rate by Sensor", "Match Rate (%)")
    
    ax.grid(True, axis="x", linestyle=":", alpha=0.4, color="#CCCCCC", zorder=0)
    ax.set_axisbelow(True)
    ax.set_xlim(0, 108)
    
    # Labels on bars
    for bar in bars:
        width = bar.get_width()
        ax.text(width + 1.8, bar.get_y() + bar.get_height()/2, f"{width:.2f}%", 
                va="center", ha="left", color="black", fontsize=8.5, fontweight="semibold")
                
    fig.tight_layout()
    return fig


def plot_confidence_by_algorithm(frame: pd.DataFrame) -> plt.Figure:
    """Plot confidence distributions by sensor / algorithm."""
    fig, ax = plt.subplots(figsize=(7.0, 4.5), facecolor="white")
    
    sensors = sorted(frame["sensor"].dropna().unique())
    data = []
    labels = []
    for sensor in sensors:
        subset = frame[frame["sensor"] == sensor]["confidence"].dropna()
        if not subset.empty:
            data.append(subset.values)
            labels.append(f"{sensor}\n(n={len(subset):,})")
            
    if data:
        # Custom boxplot styling
        try:
            import re
            v_parts = [int(x) for x in re.findall(r"\d+", matplotlib.__version__)]
        except Exception:
            v_parts = [3, 0]
        use_tick_labels = len(v_parts) >= 2 and (v_parts[0] > 3 or (v_parts[0] == 3 and v_parts[1] >= 9))
        
        flierprops = dict(marker="o", markersize=3.0, markerfacecolor="#999999", markeredgecolor="none", alpha=0.3)
        boxprops = dict(linewidth=1.2, edgecolor="#222222")
        whiskerprops = dict(linewidth=1.0, color="#666666", linestyle="-")
        capprops = dict(linewidth=1.0, color="#666666")
        medianprops = dict(linewidth=1.5, color="black")
        
        if use_tick_labels:
            box = ax.boxplot(data, tick_labels=labels, patch_artist=True, widths=0.45,
                             flierprops=flierprops, boxprops=boxprops, whiskerprops=whiskerprops,
                             capprops=capprops, medianprops=medianprops)
        else:
            box = ax.boxplot(data, labels=labels, patch_artist=True, widths=0.45,
                             flierprops=flierprops, boxprops=boxprops, whiskerprops=whiskerprops,
                             capprops=capprops, medianprops=medianprops)
            
        # Color each box corresponding to its sensor
        for idx, patch in enumerate(box["boxes"]):
            sensor_name = sensors[idx]
            patch.set_facecolor(SENSOR_COLORS.get(sensor_name, OKABE_ITO_COLORS[idx % len(OKABE_ITO_COLORS)]))
            patch.set_alpha(0.85)
            
    _apply_premium_style(ax, "Confidence Score Distribution by Sensor", ylabel="Confidence score / value")
    ax.grid(True, axis="y", linestyle=":", alpha=0.4, color="#CCCCCC", zorder=0)
    ax.set_axisbelow(True)
    
    fig.tight_layout()
    return fig


def plot_event_concentration(frame: pd.DataFrame) -> plt.Figure:
    """Plot hotspot count by fire event to show spatial concentration."""
    fig, ax = plt.subplots(figsize=(7.0, 4.5), facecolor="white")
    
    top_events = frame.head(10).copy()
    total_matched = frame["matched_hotspots"].sum()
    
    labels = []
    for _, row in top_events.iterrows():
        name = str(row["fire_name"])
        if len(name) > 20:
            name = name[:18] + "..."
        labels.append(name)
        
    y_pos = np.arange(len(labels))
    bars = ax.barh(y_pos, top_events["matched_hotspots"], color=OKABE_ITO_COLORS[0], edgecolor="none", height=0.55)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(labels, color="black", fontsize=9)
    ax.invert_yaxis()
    
    _apply_premium_style(ax, f"Match Concentration Across Top 10 Events (Total Matched N={total_matched:,})", "Number of Matched Hotspots")
    
    ax.grid(True, axis="x", linestyle=":", alpha=0.4, color="#CCCCCC", zorder=0)
    ax.set_axisbelow(True)
    
    # Value labels
    for bar in bars:
        width = bar.get_width()
        pct = (width / total_matched) * 100.0
        ax.text(width + (total_matched * 0.01), bar.get_y() + bar.get_height()/2, 
                f"{int(width):,} ({pct:.1f}%)", 
                va="center", ha="left", color="black", fontsize=8.5, fontweight="semibold")
                
    ax.set_xlim(0, max(top_events["matched_hotspots"]) * 1.15)
    fig.tight_layout()
    return fig


def plot_pilot_map(hotspots: pd.DataFrame, polygons: List[dict], displayed_n: int) -> plt.Figure:
    """Plot map showing hotspots and fire event boundary polygons."""
    fig, ax = plt.subplots(figsize=(8.0, 7.5), facecolor="white")
    
    # 1. Plot polygons
    for feature in polygons:
        geometry = feature.get("geometry", {})
        coords = geometry.get("coordinates", [])
        geom_type = geometry.get("type")
        
        poly_list = [coords] if geom_type == "Polygon" else coords
        for poly in poly_list:
            for ring in poly:
                x = [pt[0] for pt in ring]
                y = [pt[1] for pt in ring]
                ax.plot(x, y, color="black", linewidth=1.0, alpha=0.5, zorder=2)
                ax.fill(x, y, color="#999999", alpha=0.1, zorder=1)
                
    # 2. Plot hotspots colored by sensor mapping (with distinct markers)
    matched = hotspots[hotspots["match_class"].isin(["bushfire", "prescribed_burn", "other_fire", "spatial_only"])]
    unmatched = hotspots[hotspots["match_class"] == "unmatched"]
    
    # Plot matched by sensor type
    sensors = sorted(matched["sensor"].dropna().unique())
    for sensor in sensors:
        subset = matched[matched["sensor"] == sensor]
        ax.scatter(subset["longitude"], subset["latitude"], 
                   color=SENSOR_COLORS.get(sensor, OKABE_ITO_COLORS[1]), 
                   label=f"Matched {sensor} (n={len(subset):,})", 
                   s=12, alpha=0.65, zorder=4, marker="o", edgecolors="none")
                   
    # Plot unmatched (unresolved)
    ax.scatter(unmatched["longitude"], unmatched["latitude"], 
               color=SENSOR_COLORS["unresolved"], 
               label=f"Unresolved (n={len(unmatched):,})", 
               s=8, alpha=0.45, zorder=3, marker="x")
               
    _apply_premium_style(ax, f"Active-Fire Spatiotemporal Match Map (Sample Size: {displayed_n:,})", 
                         "Longitude (WGS84)", "Latitude (WGS84)")
    
    # Zoom bounds tightly around the hotspot coordinate bounds with clean margins
    ax.set_xlim(149.3, 151.3)
    ax.set_ylim(-33.8, -31.2)
    
    ax.legend(facecolor="white", edgecolor="#CCCCCC", labelcolor="black", loc="upper right", framealpha=0.9, fontsize=8.5)
    ax.grid(True, linestyle=":", alpha=0.4, color="#CCCCCC", zorder=0)
    
    fig.tight_layout()
    return fig


## 3. Results

We load the datasets from the snapshot package and perform the spatiotemporal matching. We compare the match rates and unresolved observations between the exact matching baseline and the sensor-buffered matching pipeline.

In [ ]:
import re

def normalize_hotspot(feature: dict) -> dict:
    properties = feature.get('properties', {})
    coordinates = feature.get('geometry', {}).get('coordinates', [None, None])
    return {
        'id': properties.get('id'),
        'datetime': properties.get('datetime'),
        'longitude': properties.get('longitude', coordinates[0]),
        'latitude': properties.get('latitude', coordinates[1]),
        'sensor': properties.get('sensor'),
        'satellite': properties.get('satellite'),
        'process_algorithm': properties.get('process_algorithm'),
        'confidence': properties.get('confidence'),
        'accuracy': properties.get('accuracy'),
    }

def parse_accuracy_km(value: Optional[str]) -> float:
    if value in (None, ''):
        return 0.0
    match = re.search(r'([0-9]+(?:\.[0-9]+)?)', str(value))
    return float(match.group(1)) if match else 0.0

# Helper to map NPWS schema to standard RFS-like fields consumed by matching code
def map_npws_to_rfs(feature: dict) -> dict:
    props = feature.get('properties', {})
    return {
        'geometry': feature.get('geometry'),
        'properties': {
            'fire_id': props.get('FireNo') or str(props.get('OBJECTID')),
            'fire_name': props.get('FireName') or 'Unnamed',
            'ignition_date': props.get('StartDate'),
            'extinguish_date': props.get('EndDate'),
            'fire_type': 'bushfire' if props.get('FireType') == 1 else 'prescribed_burn',
        }
    }

# 1. Dynamic search of Kaggle/local inputs
dataset_dir = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'dea_hotspots.geojson' in files:
        dataset_dir = Path(root)
        print(f"Found dataset at: {dataset_dir}")
        break
if dataset_dir is None:
    for root, dirs, files in os.walk('../input'):
        if 'dea_hotspots.geojson' in files:
            dataset_dir = Path(root)
            print(f"Found dataset at: {dataset_dir}")
            break
if dataset_dir is None:
    curr = Path('.').resolve()
    for parent in [curr] + list(curr.parents):
        candidate = parent / 'output' / 'kaggle' / 'active-fire-pilot'
        if (candidate / 'dea_hotspots.geojson').is_file():
            dataset_dir = candidate
            print(f"Found local repository dataset at: {dataset_dir}")
            break
if dataset_dir is None:
    dataset_dir = Path('.')
    print(f"Dataset not found. Falling back to current directory: {dataset_dir}")

dea_path = dataset_dir / 'dea_hotspots.geojson'
npws_path = dataset_dir / 'npws_fire_history.geojson'

with open(dea_path) as f:
    dea_data = json.load(f)
with open(npws_path) as f:
    npws_data = json.load(f)

features = [map_npws_to_rfs(f) for f in npws_data['features']]
prepared_features = prepare_features(features)

normalized_hotspots = [normalize_hotspot(h) for h in dea_data['features']]

print('Running exact matching...')
exact_classified = [classify_hotspot(h, prepared_features, grace_days=TEMPORAL_GRACE_DAYS, spatial_buffer_km=0.0) for h in normalized_hotspots]
df_exact = pd.DataFrame(exact_classified)

print('Running sensor-buffered matching...')
buffered_classified = []
for h in normalized_hotspots:
    accuracy_val = parse_accuracy_km(h.get('accuracy'))
    buffered_classified.append(classify_hotspot(h, prepared_features, grace_days=TEMPORAL_GRACE_DAYS, spatial_buffer_km=accuracy_val))
df_buffered = pd.DataFrame(buffered_classified)

# Compute headline summary metrics
headline = headline_summary(df_exact, df_buffered)
headline['fire_event_count'] = len(features)
print('\n=== HEADLINE METRICS ===')
for k, v in headline.items():
    if 'rate' in k:
        print(f'{k}: {v * 100.0:.2f}%')
    else:
        print(f'{k}: {v:,}')

# Assert snapshot invariants if in snapshot mode
expected_invariants = {
    'total_hotspots': 19849,
    'fire_event_count': 14,
    'exact_matches': 15334,
    'buffered_matches': 19277,
    'buffered_unresolved': 572,
}
if EXECUTION_MODE == 'snapshot':
    assert_snapshot_invariants(headline, expected_invariants)
    print('\n[SUCCESS] Snapshot invariants verified successfully.')


### Interpretation of Results:
- **Exact Matching Baseline:** Containment-only matching yields a 77.25% match rate, leaving 4,515 hotspots unresolved.
- **Sensor-Buffered matching:** Expanding containment boundaries by the sensors' spatial resolution tolerances increase the match rate to 97.12%, resolving all but 572 hotspots.
- **Context:** While buffering resolves spatial uncertainty at polygon edges, the high baseline match rate (77.25%) is heavily influenced by the spatial-temporal footprints of the fire complexes in this area, which we inspect below.

## 4. Visual Analysis & Event Concentration

### The Event Concentration Problem
While the headline matching metrics show high overall alignment (77.25% exact, 97.12% buffered), a detailed inspection of the match distribution across individual fire events reveals a severe concentration pattern:

* **The Dominance of Two Mega-Complexes:** Out of the 14 fire events analyzed in this region, just two events—the **Kerry Ridge** complex (183,647 ha) and the **Gospers Mountain** complex (479,514 ha)—account for **97.85% of all matched hotspots** (with Kerry Ridge alone capturing 85.34% of exact matches and 84.97% of buffered matches).
* **Spatial Dominance:** The Gospers Mountain polygon alone covers approximately 4,795 km², representing roughly 21% of the entire 22,500 km² study bounding box.
* **The "Hold-out Event" Caveat:** Because the statistical sample is almost entirely composed of hotspots from these two massive fire complexes, the reported reliability rates are highly sensitive to their specific characteristics. They **cannot be assumed to generalize** to other regions of New South Wales or to smaller, isolated fire incidents. Any operational performance metrics derived from this pilot are effectively evaluations of these two mega-fires.

### 4.1 Sensor Composition

We plot the distribution of raw active-fire hotspots across the different satellite instruments in our snapshot.

In [ ]:
fig1 = plot_sensor_composition(pd.DataFrame([
    {'sensor': s, 'total_hotspots': count}
    for s, count in df_exact.groupby('sensor').size().items()
]))
plt.show()

In [ ]:
sensor_counts = df_exact['sensor'].value_counts()
total = len(df_exact)
pcts = (sensor_counts / total * 100).to_dict()
dominant_sensor = sensor_counts.index[0]
dominant_pct = pcts[dominant_sensor]

takeaway = f"""**Sensor Composition Takeaway:**
Geostationary {dominant_sensor} dominates the hotspot observation count (representing {dominant_pct:.1f}% of all detections) due to its high temporal update frequency (every 10 minutes). """
parts = []
for sensor, pct in sorted(pcts.items(), key=lambda x: x[1], reverse=True)[1:]:
    parts.append(f"{sensor} accounts for {pct:.1f}%")
takeaway += ", ".join(parts) + " of the total dataset."
from IPython.display import display, Markdown
display(Markdown(takeaway))

### 4.2 Match Rates by Sensor

We plot the match rate of each sensor type to evaluate spatiotemporal reliability.

In [ ]:
df_sensor = sensor_summary(df_buffered)
fig2 = plot_match_rates(df_sensor)
plt.show()

In [ ]:
df_sensor_stats = sensor_summary(df_buffered)
sensor_rates = {row['sensor']: row['match_rate'] * 100 for _, row in df_sensor_stats.iterrows()}
sorted_rates = sorted(sensor_rates.items(), key=lambda x: x[1], reverse=True)
rates_str = ", ".join([f"{sensor} at {rate:.2f}%" for sensor, rate in sorted_rates])

takeaway = f"""**Match Rates Takeaway:**
Under sensor-buffered matching thresholds, {rates_str}. The higher match rates reflect how spatial buffers compensate for nominal grid-cell sizes, especially for high-resolution polar-orbiting sensors."""
from IPython.display import display, Markdown
display(Markdown(takeaway))

### 4.3 Confidence Distribution

We evaluate the distribution of confidence values across sensors.

In [ ]:
fig3 = plot_confidence_by_algorithm(df_buffered)
plt.show()

In [ ]:
modis_conf = df_buffered[df_buffered['sensor'] == 'MODIS']['confidence'].dropna()
modis_median = modis_conf.median() if not modis_conf.empty else 0.0
modis_fixed_50 = (modis_conf == 50).mean() * 100 if not modis_conf.empty else 0.0

viirs_conf = df_buffered[df_buffered['sensor'] == 'VIIRS']['confidence'].dropna()
viirs_median = viirs_conf.median() if not viirs_conf.empty else 0.0
viirs_discrete_pct = viirs_conf.isin([7, 8, 9, 'low', 'nominal', 'high']).mean() * 100 if not viirs_conf.empty else 0.0

takeaway = f"""**Confidence Distributions Takeaway:**
The confidence distributions reveal a mixed algorithmic scaling model across sensors rather than a clean split:
- **MODIS:** Median confidence is {modis_median:.1f}%. Approximately {modis_fixed_50:.1f}% of the records are fixed at exactly 50 due to SRSS algorithm defaults, while the remaining subset uses a continuous scale.
- **VIIRS:** Uses a hybrid scheme. Approximately {viirs_discrete_pct:.1f}% of the records utilize discrete values (such as 7, 8, 9 for the AFIMG algorithm), while a continuous scale is used for other algorithm variants (with an overall median confidence value of {viirs_median:.1f})."""
from IPython.display import display, Markdown
display(Markdown(takeaway))

### 4.4 Match Concentration Across Fire Events

We evaluate the concentration of hotspot matches across the 14 fire events.

In [ ]:
df_concentration = event_concentration(df_buffered)
fig4 = plot_event_concentration(df_concentration)
plt.show()

**Event Concentration Takeaway:**
The bar chart confirms the severe concentration of matched hotspots: **Kerry Ridge** accounts for over 16,000 matches, and **Gospers Mountain** accounts for over 2,400 matches. Together they account for 97.85% of matches, highlighting the spatial-temporal scale skew.

### 4.5 Spatial Distribution Map

We plot the map showing matched and unresolved hotspots overlaid on fire boundaries.

In [ ]:
fig5 = plot_pilot_map(df_buffered, features, displayed_n=DISPLAY_SAMPLE_SIZE)
plt.show()

**Spatial Map Takeaway:**
Unresolved hotspots (marked by 'x') are mostly located outside the boundary of the reserves or on the outer edges of Gospers Mountain and Kerry Ridge. This confirms that spatial buffer thresholds resolve border matches, but active fires outside reserve boundaries remain unmatched (unresolved).

## 5. Research Implications

### Incident-Level vs. Complex-Level Confound
The transition to the NPWS Fire History layer (implemented to comply with CC BY 4.0 licensing) introduced a major methodological change:
* **NSW RFS Feature Service:** Logs granular, localized, short-duration *incident-level* records.
* **NPWS Fire History:** Represents consolidated, whole-of-season *fire-complex* boundaries (e.g., the Gospers Mountain record spans 107 active days from 25 Oct 2019 to 9 Feb 2020).

This change in source shifted the scientific unit of analysis from a localized fire front to a massive, long-lived mega-complex. Because a hotspot is statistically much more likely to intersect a 4,795 km² polygon that remains open for over three months, the observed match-rate jump (from 14.5% to 77.25% exact) is driven by this **complex-level spatial-temporal scale confound**, rather than an intrinsic improvement in the operational sensor performance or label reliability.

### Modeling Recommendations
For downstream machine learning and active-fire modeling:
1. **Validation Design:** Models evaluated on datasets dominated by mega-complexes will overfit to their specific temporal and spatial footprints. Validation frameworks must implement **split-complex cross-validation**, where entire large complexes (like Gospers Mountain) are held out during training to test generalization on smaller, independent fires.
2. **Buffer Attribution:** While spatial buffering increases the match rate to 97.12%, its role must not be overstated. Exact matching is already 77.25% because of the immense size of the target polygons; buffering only resolves minor edge-drift (accounting for a ~20% marginal increase).

### Project Context and Roadmap
This notebook is the data-quality foundation stage of a larger applied research project, not a standalone endpoint. The reliability findings above — particularly the event-concentration and reference-granularity confounds — directly inform the next stage: building a confidence-filtered, multi-decade active-fire hotspot dataset fused with auxiliary weather and vegetation/land-cover covariates, for short-horizon (1–7 day) hotspot forecasting using a multimodal spatiotemporal transformer with cross-modal attention fusion. This pilot is not a fire-prediction system; it establishes the reference-data reliability constraints that any downstream predictive model built on this data must account for.

## 6. Reproducibility

To reproduce this analysis, check the local configuration and confirmed data sources:

### Confirmed Public-Source Attributions:
- **DEA Hotspots WFS:** Provided by Geoscience Australia under CC BY 4.0. URL: [DEA Hotspots Service](https://hotspots.dea.ga.gov.au/)
- **NPWS Fire History:** Provided by NSW National Parks and Wildlife Service under Creative Commons Attribution. URL: [Data.NSW NPWS Record](https://data.nsw.gov.au/data/dataset/npws-fire-history)

### Snapshot Provenance Checksums:
- `dea_hotspots.geojson` (SHA-256): `e3fef8c1c9b4a81b07482eca2209885dc0b9c5f08fc5c6ddf310ad39313655d3`
- `npws_fire_history.geojson` (SHA-256): `990507571b2b028c0d5687a7ba4351adb0b7e60ced1a53eddf9eb30e91f92dd5`

In [ ]:
import json
snapshot = {
    "total_hotspots": int(headline["total_hotspots"]),
    "exact_matches": int(headline["exact_matches"]),
    "exact_match_rate": round(headline["exact_match_rate"], 4),
    "buffered_matches": int(headline["buffered_matches"]),
    "buffered_match_rate": round(headline["buffered_match_rate"], 4),
    "buffered_unresolved": int(headline["buffered_unresolved"]),
}
print("=== REPRODUCIBILITY SNAPSHOT ===")
print(json.dumps(snapshot, indent=2))

In [ ]:
# Live Refresh Mode (guarded)
if EXECUTION_MODE == 'live_refresh':
    # In live refresh mode, we would query the active REST services.
    # Since this is a CPU-only private Kaggle runtime with internet disabled by default,
    # any live refresh must be triggered in an authorized environment.
    print('Initializing live refresh...')
    # Rerun code goes here...
